In [0]:

# ============================================================
# CELDA 1. EDA GOLD: AEROPUERTOS, AEROLÍNEAS Y CLIMA
# ============================================================

# Analizamos el comportamiento de los retrasos según:
# 1. Aeropuerto de origen.
# 2. Aerolínea.
# 3. Precipitación pronosticada.
#
# Utilizamos los vuelos de Gold que tienen clima.
# Todas las tasas consideran retrasos de 15 minutos o más.

from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. CARGAR DATOS
# ------------------------------------------------------------

vuelos = spark.table(
    "workspace.gold_flights.flights_weather_features"
)

print("Datos Gold con clima cargados correctamente.")

# ------------------------------------------------------------
# 2. RETRASOS POR AEROPUERTO DE ORIGEN
# ------------------------------------------------------------

por_aeropuerto = (
    vuelos
    .groupBy("origin")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(
            F.avg("delayed_15") * 100, 2
        ).alias("delay_rate")
    )
    .filter(F.col("total_flights") >= 1000)
    .orderBy(F.desc("delay_rate"))
    .limit(15)
    .toPandas()
)

plt.figure(figsize=(11, 6))

plt.barh(
    por_aeropuerto["origin"],
    por_aeropuerto["delay_rate"]
)

plt.gca().invert_yaxis()
plt.title("15 aeropuertos con mayor porcentaje de retrasos")
plt.xlabel("Vuelos con retraso ≥15 min (%)")
plt.ylabel("Aeropuerto de origen")
plt.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3. RETRASOS POR AEROLÍNEA
# ------------------------------------------------------------

por_aerolinea = (
    vuelos
    .groupBy("airline")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(
            F.avg("delayed_15") * 100, 2
        ).alias("delay_rate")
    )
    .filter(F.col("total_flights") >= 1000)
    .orderBy(F.desc("delay_rate"))
    .toPandas()
)

plt.figure(figsize=(11, 6))

plt.bar(
    por_aerolinea["airline"],
    por_aerolinea["delay_rate"]
)

plt.title("Porcentaje de vuelos retrasados por aerolínea")
plt.xlabel("Código de aerolínea")
plt.ylabel("Vuelos con retraso ≥15 min (%)")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4. RETRASOS SEGÚN PRECIPITACIÓN PRONOSTICADA
# ------------------------------------------------------------

# Formamos intervalos de precipitación en mm.
# Son categorías descriptivas para comparar tasas.

vuelos_clima = vuelos.withColumn(
    "rain_category",
    F.when(
        F.col("precipitation") == 0,
        "Sin lluvia"
    )
    .when(
        F.col("precipitation") < 1,
        "Mayor a 0 y menor a 1 mm"
    )
    .when(
        F.col("precipitation") < 5,
        "1 a menos de 5 mm"
    )
    .when(
        F.col("precipitation") < 10,
        "5 a menos de 10 mm"
    )
    .otherwise("10 mm o más")
)

por_lluvia = (
    vuelos_clima
    .groupBy("rain_category")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(
            F.avg("delayed_15") * 100, 2
        ).alias("delay_rate")
    )
    .toPandas()
)

orden_lluvia = [
    "Sin lluvia",
    "Mayor a 0 y menor a 1 mm",
    "1 a menos de 5 mm",
    "5 a menos de 10 mm",
    "10 mm o más"
]

por_lluvia["rain_category"] = por_lluvia[
    "rain_category"
].astype(str)

por_lluvia["rain_category"] = (
    __import__("pandas").Categorical(
        por_lluvia["rain_category"],
        categories=orden_lluvia,
        ordered=True
    )
)

por_lluvia = por_lluvia.sort_values(
    "rain_category"
)

plt.figure(figsize=(12, 5))

plt.bar(
    por_lluvia["rain_category"].astype(str),
    por_lluvia["delay_rate"]
)

plt.title("Retrasos según precipitación pronosticada")
plt.xlabel("Precipitación horaria pronosticada")
plt.ylabel("Vuelos con retraso ≥15 min (%)")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. MOSTRAR RESULTADOS
# ------------------------------------------------------------

print("=" * 55)
print("EDA GOLD: AEROPUERTOS, AEROLÍNEAS Y CLIMA")
print("=" * 55)

print("\nAEROPUERTOS CON MAYOR TASA DE RETRASOS")
display(por_aeropuerto.head(10))

print("\nAEROLÍNEAS CON MAYOR TASA DE RETRASOS")
display(por_aerolinea.head(10))

print("\nRETRASOS SEGÚN PRECIPITACIÓN")
display(por_lluvia)

print("\nEDA GOLD DE DIEGO COMPLETADO.")


In [0]:

# ============================================================
# CELDA 2. VALIDACIÓN DEL ANÁLISIS DE PRECIPITACIÓN
# ============================================================

# Revisamos el tamaño de cada grupo para comprobar
# si las diferencias en las tasas son representativas.

from pyspark.sql import functions as F

resumen_lluvia = (
    vuelos_clima
    .groupBy("rain_category")
    .agg(
        F.count("*").alias("total_vuelos"),
        F.sum("delayed_15").alias("vuelos_retrasados"),
        F.round(
            F.avg("delayed_15") * 100,
            2
        ).alias("porcentaje_retrasos")
    )
    .orderBy("rain_category")
)

print("VALIDACIÓN DE RETRASOS SEGÚN PRECIPITACIÓN")
display(resumen_lluvia)
